# Pattern ② HTTP API連携

DifyのHTTPリクエストノードを使って、Databricks REST APIを直接呼び出します。

## Databricksが提供するAPI一覧

| API | エンドポイント | 形式 | Dify連携 |
|-----|-------------|------|----------|
| UC Functions | SQL Statements API | REST (同期) | ◎ HTTPノードで直接呼べる |
| Vector Search | Vector Search API | REST (同期) | ◎ HTTPノードで直接呼べる |
| Genie | Conversation API | REST (非同期) | △ ポーリング必要 |
| KA / MAS / Agent | Serving Endpoints | ResponsesAgent形式 | △ input/output形式 |

## アーキテクチャ
```
Dify Workflow
  │
  ├── HTTPリクエスト ──▶ SQL Statements API（UC関数実行）
  ├── HTTPリクエスト ──▶ Vector Search API（セマンティック検索）
  ├── HTTPリクエスト ──▶ Genie API（自然言語→SQL）
  └── HTTPリクエスト ──▶ Serving Endpoints（KA/MAS/Agent呼び出し）
```

In [ ]:
%run ./config

In [ ]:
import requests
import json

host = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}
print(f"Workspace: https://{host}")

---
## 1. SQL Statements API（UC関数の実行）

UC関数をSQL経由で呼び出します。Difyからは最もシンプルに使えるパターンです。

In [ ]:
# ウェアハウスID取得
warehouses = requests.get(f"https://{host}/api/2.0/sql/warehouses", headers=headers).json()
warehouse_id = warehouses.get("warehouses", [{}])[0].get("id", "")
print(f"Warehouse ID: {warehouse_id}")

In [ ]:
# UC関数をSQL経由で呼び出し
sql_url = f"https://{host}/api/2.0/sql/statements/"
sql_payload = {
    "warehouse_id": warehouse_id,
    "statement": f"SELECT * FROM {catalog}.{schema_code}.search_policy('返品')",
    "wait_timeout": "30s"
}

response = requests.post(sql_url, headers=headers, json=sql_payload)
result = response.json()

if result.get("status", {}).get("state") == "SUCCEEDED":
    columns = [c["name"] for c in result["manifest"]["schema"]["columns"]]
    print("✅ UC関数実行成功")
    print("カラム:", columns)
    for row in result["result"]["data_array"][:3]:
        print(dict(zip(columns, row)))
else:
    print("❌ エラー:", result)

In [ ]:
# Python UC関数のテスト（calculate_math_expression）
sql_payload_math = {
    "warehouse_id": warehouse_id,
    "statement": f"SELECT {catalog}.{schema_code}.calculate_math_expression('sqrt(2 + 3 * 4)')",
    "wait_timeout": "30s"
}

math_resp = requests.post(sql_url, headers=headers, json=sql_payload_math)
math_result = math_resp.json()

if math_result.get("status", {}).get("state") == "SUCCEEDED":
    value = math_result["result"]["data_array"][0][0]
    print(f"✅ Python UC関数: sqrt(2 + 3 * 4) = {value}")
else:
    print("❌ エラー:", math_result)

In [ ]:
# Dify HTTPリクエストノード設定値
print("=" * 60)
print("Dify HTTPリクエストノード設定（SQL API）")
print("=" * 60)
print(f"\n【Method】POST")
print(f"【URL】https://{host}/api/2.0/sql/statements/")
print(f"\n【認証】")
print(f"  タイプ: API Key")
print(f"  Auth type: Bearer")
print(f"  API Key: <PAT_TOKEN>")
print(f"\n【Headers】")
print(f"  Content-Type: application/json")
print(f"\n【Body (JSON)】")
body = {
    "warehouse_id": warehouse_id,
    "statement": f"SELECT * FROM {catalog}.{schema_code}.search_policy('{{{{#start.input#}}}}')",
    "wait_timeout": "30s"
}
print(json.dumps(body, indent=2, ensure_ascii=False))
print(f"\n【Difyワークフロー構成】")
print("  Start → Code（SQL構築）→ HTTP（API実行）→ LLM（結果整形）→ End")
print("  ※ 複数UC関数を切り替える場合、Codeノードで動的にSQL文を組み立てる")


---
## 2. Vector Search API（セマンティック検索）

Day 1で作成したVector Searchインデックスに対してREST APIで検索します。

In [ ]:
vs_index_name = f"{catalog}.{schema_code}.knowledge_base_vs_index"
vs_api_url = f"https://{host}/api/2.0/vector-search/indexes/{vs_index_name}/query"

vs_payload = {
    "query_text": "SmartWatch X100のバッテリー持続時間",
    "columns": ["content", "doc_uri"],
    "num_results": 3
}

vs_response = requests.post(vs_api_url, headers=headers, json=vs_payload)
vs_result = vs_response.json()

if "result" in vs_result:
    print("✅ Vector Search API 応答:")
    for row in vs_result["result"].get("data_array", [])[:3]:
        print(f"  Score: {row[-1]:.3f}")
        print(f"  Content: {row[0][:100]}...")
        print()
else:
    print("❌ エラー:", vs_result)

In [ ]:
print("=" * 60)
print("Dify HTTPリクエストノード設定（Vector Search）")
print("=" * 60)
print(f"\n【Method】POST")
print(f"【URL】https://{host}/api/2.0/vector-search/indexes/{vs_index_name}/query")
print(f"\n【認証】")
print(f"  タイプ: API Key")
print(f"  Auth type: Bearer")
print(f"  API Key: <PAT_TOKEN>")
print(f"\n【Headers】")
print(f"  Content-Type: application/json")
print(f"\n【Body (JSON)】")
vs_body = {
    "query_text": "{{#start.input#}}",
    "columns": ["content", "doc_uri"],
    "num_results": 3
}
print(json.dumps(vs_body, indent=2, ensure_ascii=False))
print(f"\n【Difyワークフロー構成】")
print("  Start → HTTP（検索実行）→ LLM（結果整形）→ End")


---
## 3. Genie Conversation API（自然言語→SQL）

Genie SpaceはREST APIで外部から呼び出せます。  
**非同期API**のため、会話開始→ポーリング→結果取得の3ステップが必要です。

| ステップ | メソッド | パス |
|---------|---------|------|
| 会話開始 | POST | `/api/2.0/genie/spaces/{space_id}/start-conversation` |
| 結果確認 | GET | `/api/2.0/genie/spaces/{space_id}/conversations/{conv_id}/messages/{msg_id}` |
| データ取得 | GET | `...messages/{msg_id}/query-result/{attachment_id}` |

### Difyでの実装上の注意

| 課題 | 対策 |
|------|------|
| 非同期APIでポーリングが必要 | Codeノード内で `urllib.request` + `time.sleep` ループ |
| サンドボックスのデフォルトタイムアウトが15秒 | `.env` で `SANDBOX_WORKER_TIMEOUT=120` に変更 |
| サンドボックスからのSSL証明書エラー | `ssl.CERT_NONE` でスキップ |
| レート制限 | 5 POST/分/ワークスペース（GETはカウント外） |

**Difyワークフロー構成**:
```
Start → HTTP（会話開始）→ Code（ポーリング+結果抽出）→ End
```


In [ ]:
import time

# Day 1 No-Codeで作成したGenie SpaceのIDを指定
GENIE_SPACE_ID = "01f120dd5aa41a8797fba5d0d9bbadca"  # Customer Support and Sales Analytics

# Step 1: 会話開始
genie_url = f"https://{host}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/start-conversation"
genie_resp = requests.post(genie_url, headers=headers, json={"content": "How many products do we have?"})
genie_data = genie_resp.json()

conv_id = genie_data["conversation_id"]
msg_id = genie_data["message_id"]
print(f"✅ 会話開始: conv={conv_id}, msg={msg_id}")

# Step 2: ポーリングで結果を待つ
poll_url = f"https://{host}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/conversations/{conv_id}/messages/{msg_id}"
for i in range(20):
    time.sleep(3)
    poll_resp = requests.get(poll_url, headers=headers).json()
    status = poll_resp.get("status", "UNKNOWN")
    print(f"  Polling {i+1}: {status}")
    if status in ["COMPLETED", "FAILED", "CANCELLED"]:
        break

# Step 3: 結果表示
if status == "COMPLETED":
    attachments = poll_resp.get("attachments", [])
    for att in attachments:
        # text はstr or dictの場合がある
        text = att.get("text", "")
        if isinstance(text, dict):
            text = text.get("content", str(text))
        if text:
            print(f"\n✅ Genie回答: {str(text)[:300]}")
        # query もstr or dictの場合がある
        query = att.get("query", {})
        if isinstance(query, dict) and query.get("query"):
            print(f"\n  生成SQL: {str(query['query'])[:200]}")
else:
    print(f"❌ Genie応答失敗: {status}")
    print(json.dumps(poll_resp, indent=2, ensure_ascii=False)[:300])

---
## 4. Serving Endpoints（KA / MAS / Day1 Agent）

Knowledge Assistant、Supervisor Agent、Day 1のCode Agentは全てModel Servingエンドポイントとして公開されます。  
**ResponsesAgent形式**（`input`/`output`）で呼び出します。

| エンドポイント | 種類 | 説明 |
|-------------|------|------|
| `ka-aadba74d-endpoint` | Knowledge Assistant | ドキュメントQ&A |
| `mas-ca5427c7-endpoint` | Supervisor Agent | マルチエージェント |
| `ricoh_technova_agent_yao_sl_st_catalog_ricoh_handson_code` | Code Agent | Day 1 LangGraphエージェント |

> **注意**: これらはOpenAI Chat Completions形式（`messages`/`choices`）ではなく、  
> ResponsesAgent形式（`input`/`output`）を使います。  
> そのためDifyの「LLMモデル」としては接続できませんが、HTTPリクエストノードで呼び出せます。

In [ ]:
# Knowledge Assistant を呼び出し
ka_endpoint = "ka-aadba74d-endpoint"
ka_url = f"https://{host}/serving-endpoints/{ka_endpoint}/invocations"
ka_payload = {
    "input": [{"role": "user", "content": "What is the return policy?"}]
}

ka_resp = requests.post(ka_url, headers=headers, json=ka_payload)
ka_result = ka_resp.json()

# ResponsesAgent形式: output[-1].content[-1]["text"]
try:
    answer = ka_result["output"][-1]["content"][-1]["text"]
    print(f"✅ KA応答:\n{answer[:300]}")
except (KeyError, IndexError, TypeError):
    print(json.dumps(ka_result, indent=2, ensure_ascii=False)[:500])

In [ ]:
# Supervisor Agent (MAS) を呼び出し
mas_endpoint = "mas-ca5427c7-endpoint"
mas_url = f"https://{host}/serving-endpoints/{mas_endpoint}/invocations"
mas_payload = {
    "input": [{"role": "user", "content": "What products do you have?"}]
}

mas_resp = requests.post(mas_url, headers=headers, json=mas_payload)
mas_result = mas_resp.json()

try:
    answer = mas_result["output"][-1]["content"][-1]["text"]
    print(f"✅ MAS応答:\n{answer[:300]}")
except (KeyError, IndexError, TypeError):
    print(json.dumps(mas_result, indent=2, ensure_ascii=False)[:500])

In [ ]:
# Day 1 Code Agent を呼び出し
agent_endpoint = ENDPOINT_NAME  # ricoh_technova_agent_yao_sl_st_catalog_ricoh_handson_code
agent_url = f"https://{host}/serving-endpoints/{agent_endpoint}/invocations"
agent_payload = {
    "input": [{"role": "user", "content": "SmartWatch X100のバッテリーについて教えて"}]
}

agent_resp = requests.post(agent_url, headers=headers, json=agent_payload)
agent_result = agent_resp.json()

try:
    answer = agent_result["output"][-1]["content"][-1]["text"]
    print(f"✅ Day1 Agent応答:\n{answer[:300]}")
except (KeyError, IndexError, TypeError):
    print(json.dumps(agent_result, indent=2, ensure_ascii=False)[:500])

In [ ]:
print("=" * 60)
print("Dify HTTPリクエストノード設定（Serving Endpoints共通）")
print("=" * 60)
print(f"\n【Method】POST")
print(f"【URL例】")
print(f"  KA:     https://{host}/serving-endpoints/{ka_endpoint}/invocations")
print(f"  MAS:    https://{host}/serving-endpoints/{mas_endpoint}/invocations")
print(f"  Agent:  https://{host}/serving-endpoints/{agent_endpoint}/invocations")
print(f"\n【認証】")
print(f"  タイプ: API Key")
print(f"  Auth type: Bearer")
print(f"  API Key: <PAT_TOKEN>")
print(f"\n【Headers】")
print(f"  Content-Type: application/json")
print(f"\n【Body (JSON)】")
ep_body = {
    "input": [{"role": "user", "content": "{{#start.input#}}"}]
}
print(json.dumps(ep_body, indent=2, ensure_ascii=False))
print(f"\n【Difyワークフロー構成】")
print("  Start → HTTP（エンドポイント呼び出し）→ Code（回答抽出）→ End")
print("  ※ LLMノード不要（エンドポイントが自然言語で回答を返すため）")
print(f"\n【Codeノード（回答抽出）の実装】")
print("""import json\n\ndef main(response_body: str) -> dict:\n    data = json.loads(response_body)\n    output = data.get("output", [])\n    for item in reversed(output):\n        if item.get("type") == "message":\n            content = item.get("content", [])\n            for c in content:\n                if isinstance(c, dict) and "text" in c:\n                    return {"result": c["text"]}\n    return {"result": response_body[:2000]}""")


---
## まとめ: Difyから使えるDatabricks API

| API | URL パターン | リクエスト形式 | Dify連携の容易さ |
|-----|------------|-------------|----------------|
| SQL API | `/api/2.0/sql/statements/` | `{statement, warehouse_id}` | ◎ 同期・シンプル |
| Vector Search | `/api/2.0/vector-search/indexes/.../query` | `{query_text, columns}` | ◎ 同期・シンプル |
| Genie | `/api/2.0/genie/spaces/.../start-conversation` | `{content}` | △ 非同期・ポーリング必要 |
| KA/MAS/Agent | `/serving-endpoints/.../invocations` | `{input: [{role, content}]}` | ○ 同期だがResponsesAgent形式 |

### Difyワークフロー構成（API別）

| API | ワークフロー構成 | LLMノード |
|-----|----------------|----------|
| SQL API | Start → Code（SQL構築）→ HTTP → LLM（整形）→ End | **必要**（生JSONを整形） |
| Vector Search | Start → HTTP → LLM（整形）→ End | **必要**（生JSONを整形） |
| Genie | Start → HTTP（開始）→ Code（ポーリング+抽出）→ End | 不要 |
| KA/MAS/Agent | Start → HTTP → Code（回答抽出）→ End | 不要（自然言語で返る） |

### Dify HTTPリクエストノードの認証設定

Headersに `Authorization: Bearer` を手動で書くのではなく、**認証セクション**を使う：
```
認証タイプ: API Key
Auth type:  Bearer
API Key:    <PAT_TOKEN>
```

### ポイント
- 全API共通で `Authorization: Bearer <PAT>` で認証
- UC関数 / Vector Search は**同期API**でDifyのHTTPノードと相性が良い
- KA / MAS / Code Agentは**ResponsesAgent形式**（`input`/`output`）— LLMプロバイダーではなくHTTPツールとして使う
- GenieはDifyのCodeノード内でポーリングを実装（`SANDBOX_WORKER_TIMEOUT=120`が必要）
- ResponsesAgentのレスポンスには `function_call`, `function_call_output`, `message` が混在 — 最終回答は `type=message` のみ


---
## 補足: 認証・権限管理の仕組み

### 全パターン共通

全APIは **PAT（Personal Access Token）** で認証します。PATからユーザー/SPを特定し、Unity Catalogで権限チェックします。

```
Dify HTTP Node → Bearer <PAT> → Databricks → PATでユーザー特定 → UC権限チェック
```

### パターン別の権限チェック

| パターン | 必要なもの | 権限チェック |
|---------|----------|------------|
| SQL API（UC関数） | PAT + warehouse_id | Warehouse CAN USE + UC EXECUTE FUNCTION |
| Vector Search | PAT | UC SELECT on index |
| Genie | PAT + space_id | Space CAN VIEW |
| KA/MAS/Agent | PAT | Endpoint CAN QUERY |

### Dify運用時のポイント

Difyアプリには**固定のPAT**が埋め込まれるため、全ユーザーが同じ権限でアクセスします。

```
User A ─┐                   固定PAT
User B ─┼─▶ Dify App ──────────────▶ Databricks
User C ─┘   （全員同じ権限）         UC権限で制御
```

| 要件 | 推奨方式 |
|------|--------|
| 全員同じデータでOK | Dify + 固定SP（シンプル） |
| 部門別にデータを分けたい | Dify App × SP を分離 |
| ユーザー単位で厳格に制御 | **Databricks Apps**（OAuth U2M） |

> **ベストプラクティス**: Dify用のサービスプリンシパル（SP）を作成し、必要最小限のUC権限をGRANT。  
> 個人PATではなくSPのPATを使うことで、退職・異動の影響を受けない運用が可能。